In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from scipy.stats.mstats import winsorize
import warnings
warnings.filterwarnings('ignore')

class AssetPricingPipeline:
    def __init__(self, db_path='tail_risk_data.db'):
        self.db_path = db_path
        self.df = None
        self.feature_cols = []
        
    def load_and_build_features(self):
        """
        从 SQLite 极速读取，并执行严格的特征工程和目标标准化
        """
        print("Loading data from SQLite database...")
        conn = sqlite3.connect(self.db_path)
        # SQLite 存储的时间是字符串，使用 parse_dates 直接转换为 datetime
        df_micro = pd.read_sql("SELECT * FROM micro_data", conn, parse_dates=['date', 'datadate', 'linkdt', 'linkenddt'])
        df_macro = pd.read_sql("SELECT * FROM macro_data", conn, parse_dates=['date'])
        conn.close()
        
        print("Executing Feature Engineering...")
        # 确保排序，这对时序计算至关重要
        df = df_micro.sort_values(['permno', 'date']).reset_index(drop=True)
        
        # --- 1. 底层截面特征 ---
        df['Size'] = np.log(df['me'])
        df['BM'] = df['ceq'] / df['me'] 
        df['BM'] = np.where(df['BM'] < 0, np.nan, df['BM'])
        df['OP'] = (df['revt'] - df['cogs'].fillna(0) - df['xsga'].fillna(0) - df['xint'].fillna(0)) / df['ceq']
        
        df['at_lag1'] = df.groupby('permno')['at'].shift(1) 
        df['INV'] = (df['at'] - df['at_lag1']) / df['at_lag1']
        df['MOM1m'] = df.groupby('permno')['ret'].shift(1)
        
        df['log_ret'] = np.log(1 + df['ret'])
        df['MOM12m'] = df.groupby('permno')['log_ret'].apply(
            lambda x: x.shift(2).rolling(window=11).sum()
        ).reset_index(level=0, drop=True)
        df['MOM12m'] = np.exp(df['MOM12m']) - 1
        
        # 与宏观数据合并
        df = pd.merge(df, df_macro, on='date', how='inner')
        
        # --- 2. 目标收益率构建 (Target Normalization) ---
        # 预测下一期收益率
        df['target_ret_raw'] = df.groupby('permno')['ret'].shift(-1)
        
        # 用 VIX 缩放目标收益率 (消除宏观异方差)
        df['target_ret_scaled'] = df['target_ret_raw'] / (df['VIX'] / 100 / np.sqrt(12))
        
        # 截面缩尾 (Winsorize) - 防止极值破坏 MDN 的 NLL 损失
        def cs_winsorize(group):
            if len(group.dropna()) > 50:
                return winsorize(group, limits=[0.01, 0.01])
            return group
        
        df['target_ret_final'] = df.groupby('date')['target_ret_scaled'].transform(cs_winsorize)
        
        # --- 3. 张量特征展开与截面秩标准化 ---
        micro_feats = ['Size', 'BM', 'OP', 'INV', 'MOM1m', 'MOM12m']
        macro_feats = ['TERM', 'DEF'] 
        
        tensor_cols = []
        for micro in micro_feats:
            norm_col = f'{micro}_norm'
            df[norm_col] = df.groupby('date')[micro].transform(
                lambda x: (x.rank() - 1) / (len(x.dropna()) - 1) * 2 - 1
            ).fillna(0) 
            tensor_cols.append(norm_col)
            
            # 张量交互: 允许 Beta 时变
            for macro in macro_feats:
                interact_name = f'{norm_col}_x_{macro}'
                df[interact_name] = df[norm_col] * df[macro]
                tensor_cols.append(interact_name)
                
        self.feature_cols = tensor_cols
        # 清除无法作为训练集的行（缺乏目标或特征）
        self.df = df.dropna(subset=['target_ret_final'] + self.feature_cols)
        print(f"Pipeline ready. Total Tensor Features: {len(self.feature_cols)}")

    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        """
        纯样本外扩展窗口生成器
        """
        dates = np.sort(self.df['date'].unique())
        
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df['date'].isin(train_dates)]
            val_data = self.df[self.df['date'].isin(val_dates)]
            test_data = self.df[self.df['date'].isin(test_dates)]
            
            # 修复：使用 pd.Timestamp() 包装 numpy.datetime64 对象
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 每次跑模型时执行此流
# ==========================================
if __name__ == "__main__":
    pipeline = AssetPricingPipeline(db_path='tail_risk_data.db')
    pipeline.load_and_build_features()
    
    generator = pipeline.expanding_window_generator(initial_train_years=20, val_years=2, test_years=1)
    
    print("\n--- Expanding Window Yielding ---")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(generator):
        print(f"Window {window_idx + 1} | Train: {info['train'][1]} | Val: {info['val'][1]} | Test: {info['test'][1]}")
        
        # 提取模型输入
        X_train = train_df[pipeline.feature_cols].values
        Y_train = train_df['target_ret_final'].values
        
        # 测试打印一轮即可
        if window_idx == 0:
            print(f"Sample X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
            break

Loading data from SQLite database...
Executing Feature Engineering...
Pipeline ready. Total Tensor Features: 18

--- Expanding Window Yielding ---
Window 1 | Train: 2009-12 | Val: 2011-12 | Test: 2012-12
Sample X_train shape: (1389838, 18), Y_train shape: (1389838,)


In [3]:
pipeline = AssetPricingPipeline(db_path='tail_risk_data.db')
pipeline.load_and_build_features()

Loading data from SQLite database...
Executing Feature Engineering...
Pipeline ready. Total Tensor Features: 18


In [5]:
# show all columns
pd.set_option('display.max_columns', None)
pipeline.df.head()

,permno,date,ret,prc,shrout,vol,exchcd,shrcd,me,gvkey,datadate,at,ceq,ni,revt,cogs,xsga,xint,linkdt,linkenddt,Size,BM,OP,at_lag1,INV,MOM1m,log_ret,MOM12m,TERM,DEF,VIX,target_ret_raw,target_ret_scaled,target_ret_final,Size_norm,Size_norm_x_TERM,Size_norm_x_DEF,BM_norm,BM_norm_x_TERM,BM_norm_x_DEF,OP_norm,OP_norm_x_TERM,OP_norm_x_DEF,INV_norm,INV_norm_x_TERM,INV_norm_x_DEF,MOM1m_norm,MOM1m_norm_x_TERM,MOM1m_norm_x_DEF,MOM12m_norm,MOM12m_norm_x_TERM,MOM12m_norm_x_DEF
0,10001,1990-01-31,-0.018519,-9.9375,1022.0,353.0,3,11,10156.125,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,9.225832,NaN,NaN,NaN,NaN,NaN,-0.018693,NaN,0.43,0.95,25.36,-0.006289,-0.085906,-0.085906,-0.459404,-0.197544,-0.436434,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,10001,1990-02-28,-0.006289,-9.8750,1022.0,149.0,3,11,10092.250,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,9.219523,NaN,NaN,NaN,NaN,-0.018519,-0.006309,NaN,0.47,0.92,21.99,0.012658,0.199402,0.199402,-0.471154,-0.221442,-0.433462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.308867,0.145167,0.284158,0.0,0.0,0.0
2,10001,1990-03-31,0.012658,-9.8750,1027.0,127.0,3,11,10141.625,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,9.224404,NaN,NaN,NaN,NaN,-0.006289,0.012579,NaN,0.58,0.84,19.73,0.000000,0.000000,0.000000,-0.473052,-0.274370,-0.397364,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.225773,-0.130948,-0.189649,0.0,0.0,0.0
3,10001,1990-04-30,0.000000,-9.8750,1027.0,166.0,3,11,10141.625,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,9.224404,NaN,NaN,NaN,NaN,0.012658,0.000000,NaN,0.97,0.84,19.52,-0.012658,-0.224634,-0.224634,-0.464126,-0.450203,-0.389866,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.059561,0.057774,0.050031,0.0,0.0,0.0
4,10001,1990-05-31,-0.012658,9.7500,1027.0,279.0,3,11,10013.250,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,9.211664,NaN,NaN,NaN,NaN,0.000000,-0.012739,NaN,0.59,0.94,17.37,0.014103,0.281256,0.281256,-0.476052,-0.280870,-0.447488,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.311999,0.184079,0.293279,0.0,0.0,0.0


In [ ]:
pipeline.df.sample(10)

,permno,date,ret,prc,shrout,vol,exchcd,shrcd,me,gvkey,datadate,at,ceq,ni,revt,cogs,xsga,xint,linkdt,linkenddt,Size,BM,OP,at_lag1,INV,MOM1m,log_ret,MOM12m,TERM,DEF,VIX,target_ret_raw,target_ret_scaled,target_ret_final,Size_norm,Size_norm_x_TERM,Size_norm_x_DEF,BM_norm,BM_norm_x_TERM,BM_norm_x_DEF,OP_norm,OP_norm_x_TERM,OP_norm_x_DEF,INV_norm,INV_norm_x_TERM,INV_norm_x_DEF,MOM1m_norm,MOM1m_norm_x_TERM,MOM1m_norm_x_DEF,MOM12m_norm,MOM12m_norm_x_TERM,MOM12m_norm_x_DEF
1841020,89546,2018-11-30,0.022280,19.73,13117.0,7396.0,1,11,258798.41,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.463805,NaN,NaN,NaN,NaN,-0.085308,0.022035,-0.241008,0.64,1.00,18.07,-0.166751,-3.196693,-3.196693,-0.349792,-0.223867,-0.349792,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.114846,0.073501,0.114846,-0.668862,-0.428072,-0.668862
2014660,92808,2020-03-31,-0.347500,13.05,12645.0,41983.0,3,11,165017.25,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.013805,NaN,NaN,NaN,NaN,-0.361226,-0.426944,0.037787,0.59,1.27,53.54,0.283525,1.834440,1.834440,-0.351215,-0.207217,-0.446044,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,-0.962338,-0.567780,-1.222170,0.150194,0.088615,0.190746
1559858,84808,2007-10-31,-0.042746,46.58,22939.0,33170.0,1,11,1068498.62,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,13.881765,NaN,NaN,NaN,NaN,0.061287,-0.043687,0.440010,0.54,0.82,18.53,-0.045943,-0.858884,-0.858884,0.380911,0.205692,0.312347,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.445198,0.240407,0.365062,0.703913,0.380113,0.577209
1536876,84381,2015-11-30,-0.018276,106.44,132015.0,243091.0,1,11,14051676.60,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,16.458252,NaN,NaN,NaN,NaN,0.075786,-0.018445,-0.076543,1.99,1.40,16.13,-0.035983,-0.772776,-0.772776,0.848846,1.689204,1.188385,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.228541,0.454796,0.319957,-0.038133,-0.075885,-0.053386
254572,14677,2021-06-30,-0.043993,5.65,97613.0,215239.0,3,11,551513.45,020540,2020-12-31,458.061,382.782,76.544,278.678,20.949,238.709,0.000,2014-05-16,NaT,13.220422,0.000694,0.049689,NaN,NaN,0.284783,-0.044990,0.703701,1.40,0.65,15.83,-0.069027,-1.510528,-1.510528,-0.164631,-0.230484,-0.107010,0.517112,0.723956,0.336123,-0.150587,-0.210822,-0.097882,0.0,0.0,0.0,0.936068,1.310496,0.608445,0.219267,0.306974,0.142524
758346,59686,1992-06-30,0.000000,33.50,36288.0,14583.0,1,11,1215648.00,002527,1991-12-31,1478.871,642.151,61.076,1607.642,1163.165,201.578,30.568,1972-12-14,1996-03-29,14.010788,0.000528,0.330656,NaN,NaN,-0.039429,0.000000,0.138366,3.49,0.83,13.35,0.003731,0.096813,0.096813,0.822694,2.871201,0.682836,-0.164912,-0.575544,-0.136877,0.534919,1.866869,0.443983,0.0,0.0,0.0,-0.345651,-1.206322,-0.286890,0.067605,0.235940,0.056112
1950229,91357,2016-10-31,-0.135703,5.35,16402.0,6552.0,3,11,87750.70,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,11.382255,NaN,NaN,NaN,NaN,0.041211,-0.145839,-0.027197,1.50,0.87,17.06,0.140187,2.846553,2.846553,-0.597799,-0.896699,-0.520085,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.417200,0.625800,0.362964,-0.315927,-0.473890,-0.274856
851683,68398,2001-03-31,0.000000,3.50,8139.0,1336.0,2,11,28486.50,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,10.257186,NaN,NaN,NaN,NaN,-0.002849,0.000000,0.059622,0.63,0.86,28.64,0.005714,0.069113,0.069113,-0.479049,-0.301801,-0.411982,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.249268,0.157039,0.214370,0.184439,0.116196,0.158617
776330,61496,2007-03-31,0.045792,11.99,4912.0,309.0,3,11,58894.88,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,10.983509,NaN,NaN,NaN,NaN,-0.022592,0.044774,0.085607,-0.39,0.97,14.64,-0.019183,-0.453906,-0.453906,-0.717107,0.279672,-0.695593,0.000000,-0.000000,0.000000,0.000000,-0.000000,0.000000,0.0,-0.0,0.0,-0.227066,0.088556,-0.220254,0.097713,-0.038108,0.094782
1201279,79254,2002-02-28,0.008725,20.81,82601.0,17520.0,3,11,1718926.81,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,14

In [13]:
pipeline.feature_cols

['Size_norm',
 'Size_norm_x_TERM',
 'Size_norm_x_DEF',
 'BM_norm',
 'BM_norm_x_TERM',
 'BM_norm_x_DEF',
 'OP_norm',
 'OP_norm_x_TERM',
 'OP_norm_x_DEF',
 'INV_norm',
 'INV_norm_x_TERM',
 'INV_norm_x_DEF',
 'MOM1m_norm',
 'MOM1m_norm_x_TERM',
 'MOM1m_norm_x_DEF',
 'MOM12m_norm',
 'MOM12m_norm_x_TERM',
 'MOM12m_norm_x_DEF']

In [9]:
pipeline.df[pipeline.df['permno'] == 91357]

,permno,date,ret,prc,shrout,vol,exchcd,shrcd,me,gvkey,datadate,at,ceq,ni,revt,cogs,xsga,xint,linkdt,linkenddt,Size,BM,OP,at_lag1,INV,MOM1m,log_ret,MOM12m,TERM,DEF,VIX,target_ret_raw,target_ret_scaled,target_ret_final,Size_norm,Size_norm_x_TERM,Size_norm_x_DEF,BM_norm,BM_norm_x_TERM,BM_norm_x_DEF,OP_norm,OP_norm_x_TERM,OP_norm_x_DEF,INV_norm,INV_norm_x_TERM,INV_norm_x_DEF,MOM1m_norm,MOM1m_norm_x_TERM,MOM1m_norm_x_DEF,MOM12m_norm,MOM12m_norm_x_TERM,MOM12m_norm_x_DEF
1950105,91357,2006-06-30,NaN,17.20,20867.0,48453.0,3,11,358912.40,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.790834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.14,0.89,13.08,0.094186,2.494418,2.494418,0.002117,0.000296,0.001884,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1950106,91357,2006-07-31,0.094186,18.82,20867.0,13920.0,3,11,392716.94,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.880844,NaN,NaN,NaN,NaN,NaN,0.090011,NaN,-0.11,0.91,14.95,0.104676,2.425474,2.425474,0.055001,-0.006050,0.050051,0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,0.000000,-0.000000,0.000000,0.000000,-0.000000,0.000000
1950107,91357,2006-08-31,0.104676,20.79,20867.0,47184.0,3,11,433824.93,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.980396,NaN,NaN,NaN,NaN,0.094186,0.099552,NaN,-0.31,0.91,12.31,-0.095719,-2.693585,-2.693585,0.085423,-0.026481,0.077735,0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,0.842860,-0.261287,0.767003,0.000000,-0.000000,0.000000
1950108,91357,2006-09-30,-0.095719,18.80,20867.0,84869.0,3,11,392299.60,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.879781,NaN,NaN,NaN,NaN,0.104676,-0.100615,NaN,-0.25,0.92,11.98,0.125000,3.614463,3.614463,0.044199,-0.011050,0.040663,0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,0.663523,-0.165881,0.610441,0.000000,-0.000000,0.000000
1950109,91357,2006-10-31,0.125000,21.15,20867.0,89144.0,3,11,441337.05,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,12.997564,NaN,NaN,NaN,NaN,-0.095719,0.117783,NaN,-0.47,0.91,11.10,0.066194,2.065790,2.065790,0.069926,-0.032865,0.063632,0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,-0.766702,0.360350,-0.697699,0.000000,-0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1950280,91357,2021-01-31,0.197133,3.34,16883.0,49329.0,3,11,56389.22,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,10.940033,NaN,NaN,NaN,NaN,0.060836,0.179930,-0.403629,1.05,0.79,33.09,0.221557,2.319420,2.319420,-0.842846,-0.884988,-0.665848,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.040566,0.042595,0.032047,-0.794326,-0.834043,-0.627518
1950281,91357,2021-02-28,0.221557,4.08,16883.0,8744.0,3,11,68882.64,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,11.140159,NaN,NaN,NaN,NaN,0.197133,0.200126,-0.298996,1.40,0.72,27.95,0.279412,3.463011,3.463011,-0.828841,-1.160377,-0.596765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.639989,0.895985,0.460792,-0.778892,-1.090448,-0.560802
1950282,91357,2021-03-31,0.279412,5.22,16883.0,111944.0,3,11,88129.26,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,11.386560,NaN,NaN,NaN,NaN,0.221557,0.246401,0.027692,1.71,0.70,19.40,0.005747,0.102620,0.102620,-0.789124,-1.349403,-0.552387,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707577,1.209957,0.495304,-0.409653,-0.700506,-0.286757
1950283,91357,2021-04-30,0.005747,5.25,16883.0,24449.0,3,11,88635.75,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,11.392291,NaN,NaN,NaN,NaN,0.279412,0.005731,0.863013,1.64,0.70,18.61,0.007619,0.141822,0.141822,-0.783838,-1.285494,-0.548687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.937780,1.537959,0.656446,0.106911,0.175334,0.074838


In [10]:
import pandas as pd
import numpy as np

# 创建一个 1 到 15 的序列
s = pd.Series(range(1, 16)) 
# 假设第 15 行（索引14）是当前月
result = s.shift(2).rolling(window=11).sum()

print(f"第15行的计算结果: {result.iloc[14]}")
# 逻辑验证：
# shift(2) 后，第15行看到的是 13
# rolling(11) 包含：13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3
# sum = 88

第15行的计算结果: 88.0


In [11]:
result

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
5      NaN
6      NaN
7      NaN
8      NaN
9      NaN
10     NaN
11     NaN
12    66.0
13    77.0
14    88.0
dtype: float64